# Tabla 5.1 — Comparación de Datasets de Edificaciones
**Responsable:** Andrés  
**Capítulo:** 5.4 del informe `informe_upme_solar.tex`  
**Salida:** `semana_3/outputs/tables/comparacion_datasets.csv`

Este notebook consulta las colecciones `ms_buildings` y `goo_buildings` en MongoDB
y genera automáticamente la tabla comparativa de la Sección 5.4.  
Si alguna colección no existe o está vacía imprime un `WARNING` claro.

In [3]:
import sys
from pathlib import Path
import pandas as pd
import warnings
from datetime import datetime

# ── Raíz del proyecto ──────────────────────────────────────────
ROOT = Path("__file__").resolve().parent.parent.parent
sys.path.append(str(ROOT))
from config import get_db, BASE

# ── Carpeta de salida ──────────────────────────────────────────
OUTPUT_DIR = Path(BASE) / "semana_3" / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "comparacion_datasets.csv"

print(f"Conectando a MongoDB...")
db = get_db()
print(f"Base de datos: {db.name}")
print(f"Colecciones existentes: {db.list_collection_names()}")

Conectando a MongoDB...
Base de datos: upme_solar_db
Colecciones existentes: ['municipios_pdet', 'goo_buildings', 'ms_buildings']


In [4]:
# ── Helpers ────────────────────────────────────────────────────

def coleccion_ok(nombre: str) -> bool:
    """Devuelve True si la colección existe y tiene al menos 1 documento."""
    if nombre not in db.list_collection_names():
        print(f"\n⚠️  WARNING: PENDIENTE — la colección '{nombre}' NO existe en MongoDB.")
        print(f"    Verifica que el responsable haya ejecutado el script de carga.")
        return False
    n = db[nombre].count_documents({})
    if n == 0:
        print(f"\n⚠️  WARNING: PENDIENTE — la colección '{nombre}' está VACÍA.")
        print(f"    Verifica que el responsable haya ejecutado el script de carga.")
        return False
    return True


def get_stats(nombre: str) -> dict:
    """Retorna estadísticas clave de una colección."""
    if not coleccion_ok(nombre):
        responsable = "Juan Pablo" if nombre == "ms_buildings" else "Jineth"
        return {
            "total_edificaciones": f"PENDIENTE: {responsable}",
            "tamaño_en_disco_MB":  f"PENDIENTE: {responsable}",
            "tiene_atributo_area": f"PENDIENTE: {responsable}",
            "tiempo_de_carga":     f"PENDIENTE: {responsable}",
        }

    col = db[nombre]

    # Total de documentos
    total = col.count_documents({})

    # Tamaño en disco via collStats
    stats_cmd = db.command("collStats", nombre)
    size_mb = round(stats_cmd.get("storageSize", 0) / (1024 ** 2), 2)

    # Verificar si existe campo area_m2
    sample = col.find_one({"area_m2": {"$exists": True}})
    tiene_area = "Sí (area_m2)" if sample else "No"

    return {
        "total_edificaciones": total,
        "tamaño_en_disco_MB":  size_mb,
        "tiene_atributo_area": tiene_area,
        "tiempo_de_carga":     "ver log de carga",   # no se puede recuperar a posteriori
    }

print("Helpers definidos.")

Helpers definidos.


In [5]:
# ── Consultar colecciones ──────────────────────────────────────
print("=" * 55)
print("Consultando ms_buildings...")
ms_stats = get_stats("ms_buildings")

print("\nConsultando goo_buildings...")
goo_stats = get_stats("goo_buildings")

print("\nEstadísticas crudas:")
print("  ms_buildings :", ms_stats)
print("  goo_buildings:", goo_stats)

Consultando ms_buildings...

Consultando goo_buildings...

Estadísticas crudas:
  ms_buildings : {'total_edificaciones': 5258, 'tamaño_en_disco_MB': 1.29, 'tiene_atributo_area': 'Sí (area_m2)', 'tiempo_de_carga': 'ver log de carga'}
  goo_buildings: {'total_edificaciones': 2263446, 'tamaño_en_disco_MB': 410.89, 'tiene_atributo_area': 'Sí (area_m2)', 'tiempo_de_carga': 'ver log de carga'}


In [6]:
# ── Construir tabla comparativa ────────────────────────────────
#
# Campos fijos (documentados / conocidos de antemano)
AÑO_MS  = "2014–2021"
AÑO_GOO = "PENDIENTE: Jineth"   # completar cuando Jineth confirme
LIC_MS  = "ODbL"
LIC_GOO = "CC BY-4.0 / ODbL"

tabla = pd.DataFrame([
    {
        "aspecto":              "total_edificaciones",
        "microsoft":            ms_stats["total_edificaciones"],
        "google":               goo_stats["total_edificaciones"],
    },
    {
        "aspecto":              "tamaño_en_disco_MB",
        "microsoft":            ms_stats["tamaño_en_disco_MB"],
        "google":               goo_stats["tamaño_en_disco_MB"],
    },
    {
        "aspecto":              "año_imágenes",
        "microsoft":            AÑO_MS,
        "google":               AÑO_GOO,
    },
    {
        "aspecto":              "tiene_atributo_área",
        "microsoft":            ms_stats["tiene_atributo_area"],
        "google":               goo_stats["tiene_atributo_area"],
    },
    {
        "aspecto":              "licencia",
        "microsoft":            LIC_MS,
        "google":               LIC_GOO,
    },
    {
        "aspecto":              "tiempo_de_carga",
        "microsoft":            ms_stats["tiempo_de_carga"],
        "google":               goo_stats["tiempo_de_carga"],
    },
])

tabla["generado_en"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")

print("\nTabla 5.1 — Comparación de datasets")
print(tabla.to_string(index=False))


Tabla 5.1 — Comparación de datasets
            aspecto        microsoft            google          generado_en
total_edificaciones             5258           2263446 2026-05-24 16:34 UTC
 tamaño_en_disco_MB             1.29            410.89 2026-05-24 16:34 UTC
       año_imágenes        2014–2021 PENDIENTE: Jineth 2026-05-24 16:34 UTC
tiene_atributo_área     Sí (area_m2)      Sí (area_m2) 2026-05-24 16:34 UTC
           licencia             ODbL  CC BY-4.0 / ODbL 2026-05-24 16:34 UTC
    tiempo_de_carga ver log de carga  ver log de carga 2026-05-24 16:34 UTC


C:\Users\Juan Pablo\AppData\Local\Temp\ipykernel_9096\3171027781.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  tabla["generado_en"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")


In [7]:
# ── Exportar a CSV ─────────────────────────────────────────────
tabla.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\n✅ Tabla exportada → {OUTPUT_PATH}")

# ── Mostrar resumen legible ────────────────────────────────────
print("\n" + "=" * 55)
print("RESUMEN PARA EL INFORME (Tabla 5.1)")
print("=" * 55)
for _, row in tabla.iterrows():
    print(f"  {row['aspecto']:<25} | MS: {str(row['microsoft']):<30} | GOO: {row['google']}")

# ── Advertencia si hay pendientes ─────────────────────────────
pendientes = tabla[
    tabla.apply(lambda r: "PENDIENTE" in str(r["microsoft"]) or "PENDIENTE" in str(r["google"]), axis=1)
]
if not pendientes.empty:
    print("\n" + "⚠️  " * 10)
    print("CELDAS PENDIENTES EN LA TABLA:")
    for _, row in pendientes.iterrows():
        print(f"  → {row['aspecto']}: MS={row['microsoft']} | GOO={row['google']}")
    print("    Completar antes de entregar el informe.")
else:
    print("\n✅ No hay celdas pendientes — tabla completa.")


✅ Tabla exportada → C:\Users\Juan Pablo\Proyecto-DBA\semana_3\outputs\tables\comparacion_datasets.csv

RESUMEN PARA EL INFORME (Tabla 5.1)
  total_edificaciones       | MS: 5258                           | GOO: 2263446
  tamaño_en_disco_MB        | MS: 1.29                           | GOO: 410.89
  año_imágenes              | MS: 2014–2021                      | GOO: PENDIENTE: Jineth
  tiene_atributo_área       | MS: Sí (area_m2)                   | GOO: Sí (area_m2)
  licencia                  | MS: ODbL                           | GOO: CC BY-4.0 / ODbL
  tiempo_de_carga           | MS: ver log de carga               | GOO: ver log de carga

⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  ⚠️  
CELDAS PENDIENTES EN LA TABLA:
  → año_imágenes: MS=2014–2021 | GOO=PENDIENTE: Jineth
    Completar antes de entregar el informe.


In [8]:
total = db.goo_buildings.count_documents({})
print(f"Total registros: {total}")

Total registros: 2263446


In [9]:

docs = list(db.goo_buildings.find().limit(5))

for i, doc in enumerate(docs):
    print(f"\nDocumento {i+1}")
    print(doc.keys())


Documento 1
dict_keys(['_id', 'geometry', 'area_m2', 'confidence', 'cod_dane_municipio', 'fuente', 'cargado_en'])

Documento 2
dict_keys(['_id', 'geometry', 'area_m2', 'confidence', 'cod_dane_municipio', 'fuente', 'cargado_en'])

Documento 3
dict_keys(['_id', 'geometry', 'area_m2', 'confidence', 'cod_dane_municipio', 'fuente', 'cargado_en'])

Documento 4
dict_keys(['_id', 'geometry', 'area_m2', 'confidence', 'cod_dane_municipio', 'fuente', 'cargado_en'])

Documento 5
dict_keys(['_id', 'geometry', 'area_m2', 'confidence', 'cod_dane_municipio', 'fuente', 'cargado_en'])


In [10]:
import pandas as pd

docs = list(
    db.goo_buildings.find(
        {},
        {
            "area_m2": 1,
            "confidence": 1,
            "cod_dane_municipio": 1,
            "fuente": 1,
            "cargado_en": 1
        }
    )
)

df = pd.DataFrame(docs)
df.head()

,_id,area_m2,confidence,cod_dane_municipio,fuente,cargado_en
0,6a0fc2e4ecffb3fa1aecff13,43.7626,0.8335,95025,google,2026-05-22 02:43:47.554
1,6a0fc2e4ecffb3fa1aecff14,85.5311,0.8762,50590,google,2026-05-22 02:43:47.554
2,6a0fc2e4ecffb3fa1aecff15,111.9906,0.9073,95001,google,2026-05-22 02:43:47.554
3,6a0fc2e4ecffb3fa1aecff16,65.5756,0.7357,50590,google,2026-05-22 02:43:47.555
4,6a0fc2e4ecffb3fa1aecff17,8.4853,0.7166,95001,google,2026-05-22 02:43:47.555


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.strtree import STRtree
from pathlib import Path
import sys
import os

ROOT = Path("__file__").resolve().parent.parent.parent
sys.path.append(str(ROOT))
from config import BASE

csv_path = Path(BASE) / "semana_3" / "datos" / "raw" / "google" / "google_pdet_filtrado.csv"
out_path = Path(BASE) / "semana_3" / "datos" / "procesados" / "google_centroides.csv"

# Cargar municipios PDET para cruce espacial
print("Cargando municipios PDET...")
pdet = gpd.read_file(str(Path(BASE) / "semana_2" / "datos" / "procesados" / "municipios_pdet.geojson"))
tree = STRtree(pdet.geometry.values)

CHUNK_SIZE   = 50_000
total_leidos = 0
total_guardados = 0
primer_chunk = True
errores      = 0

print("Procesando CSV de Google...\n")

for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE):
    total_leidos += len(chunk)

    # Eliminar filas sin geometría
    chunk = chunk[chunk["geometry"].notna()]

    rows = []
    for _, fila in chunk.iterrows():
        try:
            geom     = wkt.loads(fila["geometry"])
            centroid = geom.centroid

            # Cruce espacial — verificar que cae dentro de un municipio PDET
            candidatos = tree.query(centroid)
            cod_dane   = None
            for idx in candidatos:
                if pdet.geometry.iloc[idx].contains(centroid):
                    cod_dane = pdet.iloc[idx]["cod_dane"]
                    break

            if cod_dane is None:
                continue

            rows.append({
                "lng":                centroid.x,
                "lat":                centroid.y,
                "area_m2":            float(fila["area_in_meters"]),
                "confidence":         float(fila["confidence"]),
                "cod_dane_municipio": cod_dane,
                "fuente":             "google"
            })
        except Exception:
            errores += 1
            continue

    if rows:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(
            str(out_path),
            mode="w" if primer_chunk else "a",
            header=primer_chunk,
            index=False
        )
        primer_chunk = False
        total_guardados += len(rows)

    print(f"  Leídos: {total_leidos:,} | Guardados: {total_guardados:,} | Errores: {errores}")

print(f"\n✅ Google centroides completado")
print(f"   Total guardados: {total_guardados:,}")
print(f"   Guardado en:     {out_path}")

Cargando municipios PDET...
Procesando CSV de Google...



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Juan Pablo\\Proyecto-DBA\\semana_3\\datos\\raw\\google\\colombia_pdet.csv'